# 옷장패션 데이터 정제 보고서

## 1. 데이터 개요
- 행/열: 1,500 × 8
- 주요 컬럼: order_id, customer_age, category, channel, price, quantity, amount, return_amount

## 2. 진단 결과
- 결측: amount 3.40%(51건), price 0.33%(5건)
- 이상치(IQR): customer_age 18건(이 중 0·999 5건만 불가능 값), quantity 1건(200),
  amount 145건(대부분 정상 고가 주문 → 과탐), return_amount 122건(0이 93% → IQR=0이라 작동 불가)
- 결측 유형: amount 결측 행의 app 비율 88% vs 전체 61% → MAR

## 3. 처리 결정과 근거
| 이슈 | 결정 | 근거 | 한계 |
| --- | --- | --- | --- |
| age 0·999 (5건) | NaN → 중앙값 대체(120은 유지 후 확인 불가 플래그) | 물리적으로 불가능 | 실제 나이 확인 불가 |
| quantity 200 (1건) | 금액 ÷ 가격으로 복원(3) | 29,900 × 3 = 89,700으로 금액과 일치 | 금액이 맞다는 가정 |
| amount 5천만 (1건) | 가격 × 수량으로 복원(49,900) | 49,900 × 1과 약 1,000배 불일치 → 입력 오류 | 원천 시스템 미확인 |
| amount 결측 (51건) | 가격 × 수량으로 복원 + 결측 플래그 | 같은 행의 가격·수량이 온전함 | 할인 등이 있었다면 오차 가능 |
| price 결측 (5건) | 금액 ÷ 수량으로 복원 | 같은 행의 금액·수량이 온전함 | 위와 같음 |
| return_amount | 처리 안 함(유효 건으로 계산) | 0은 결측이 아니라 "환불 없음" | IQR 대신 환불 비율로 따로 점검 필요 |

## 4. 처리 후 검증
- 결측 0건
- customer_age 범위: 5 ~ 60
- 모든 행에서 price × quantity = amount 성립
- fixed 플래그 2건, amount_missing 플래그 51건 보존

## 5. 후속 권고
- IQR만으로는 과탐이 심함(amount 145건) → 정합성 검사를 기본 점검에 추가
- customer_age 5살 등 어린 나이는 가능하지만 드묾 → 고객 이력 확인 권장

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

np.random.seed(42)

n_customers = 300
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(35, 9, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
    "income": np.random.choice([2500, 3500, 4500, 6000, 8500], n_customers).astype(float),
})
customers.loc[5, "age"] = 999
customers.loc[10, "age"] = -3
customers.loc[15, "age"] = 120
customers.loc[[20, 21, 22, 70, 120], "gender"] = np.nan
high_income_mask = customers["income"] >= 6000
high_idx = customers[high_income_mask].sample(frac=0.4, random_state=1).index
customers.loc[high_idx, "income"] = np.nan

categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 40
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_products),
})

n_orders = 2000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity
orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "category": products.set_index("product_id").loc[order_product, "category"].values,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app"], n_orders, p=[0.5, 0.5]),
})
app_mask = orders["channel"] == "app"
app_idx = orders[app_mask].sample(frac=0.04, random_state=2).index
web_idx = orders[~app_mask].sample(frac=0.01, random_state=3).index
orders.loc[app_idx.union(web_idx), "amount"] = np.nan
orders.loc[7, "quantity"] = 100
orders.loc[1900, "quantity"] = 50
orders.loc[1500, "amount"] = 5_000_000


def missing_summary(df):
    s = df.isnull().sum()
    p = (df.isnull().mean() * 100).round(2)
    out = pd.DataFrame({"missing": s, "missing_pct(%)": p})
    return out[out["missing"] > 0].sort_values("missing", ascending=False)


def detect_outliers_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, (lower, upper)


# 스스로 해보자! (1)
# 1) 15번 라벨 행, age 열
print(customers.loc[15, "age"])

# 2) 패션 카테고리 + 수량 2 이상 주문의 order_id, amount 앞 5행
mask = (orders["category"] == "패션") & (orders["quantity"] >= 2)
print(orders.loc[mask, ["order_id", "amount"]].head())


# 스스로 해보자! (2)
# 1) isin + notna 조합
mask = orders["category"].isin(["패션", "뷰티"]) & orders["amount"].notna()
print("건수:", mask.sum())

# 2) query로 같은 조건
print(orders.query("category in ['패션', '뷰티'] and amount.notna()", engine="python").shape)


# 스스로 해보자! (3)
# 1) orders 결측 요약
print(missing_summary(orders))

# 2) amount 결측 행의 channel 분포 vs 전체 channel 분포
null_rows = orders[orders["amount"].isnull()]
print("결측 행 채널 분포:")
print(null_rows["channel"].value_counts(normalize=True).round(2))
print("\n전체 채널 분포:")
print(orders["channel"].value_counts(normalize=True).round(2))


# 스스로 해보자! (5)
# 1) 전체 중앙값 대체
orders_a = orders.copy()
orders_a["amount"] = orders_a["amount"].fillna(orders_a["amount"].median())

# 2) 채널별 중앙값 대체
orders_b = orders.copy()
orders_b["amount"] = orders_b["amount"].fillna(
    orders_b.groupby("channel")["amount"].transform("median")
)

# 3) 채널별 평균 비교
print("[원본 — 결측 제외 평균]")
print(orders.groupby("channel")["amount"].mean().round(0))
print("[A — 전체 중앙값 대체 후]")
print(orders_a.groupby("channel")["amount"].mean().round(0))
print("[B — 채널별 중앙값 대체 후]")
print(orders_b.groupby("channel")["amount"].mean().round(0))


# 스스로 해보자! (6)
# 1) IQR 경계
mask_amt, (lo, up) = detect_outliers_iqr(orders["amount"])
print(f"하한: {lo:.0f} / 상한: {up:.0f}")

# 2) 이상치 개수·비율
n_out = mask_amt.sum()
print(f"이상치: {n_out}건 ({n_out/len(orders)*100:.2f}%)")
print(orders[mask_amt].head())


# 종합 실습 — 새 데이터셋 '옷장패션' 주문
np.random.seed(11)
n = 1500
partner = pd.DataFrame({
    "order_id": [f"K{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(33, 8, n).round().astype(int),
    "category": np.random.choice(["상의", "하의", "신발", "액세서리"], n, p=[0.35, 0.3, 0.2, 0.15]),
    "channel": np.random.choice(["web", "app"], n, p=[0.4, 0.6]),
    "price": np.random.choice([15900, 29900, 49900, 79900, 129900], n),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n),
})
partner["amount"] = partner["price"] * partner["quantity"]
partner["return_amount"] = np.where(
    np.random.rand(n) < 0.07, partner["amount"] * np.random.uniform(0.5, 1.0, n), 0
).round(0)
partner.loc[partner.sample(3, random_state=1).index, "customer_age"] = 999
partner.loc[partner.sample(2, random_state=2).index, "customer_age"] = 0
app = partner["channel"] == "app"
partner.loc[partner[app].sample(frac=0.05, random_state=3).index, "amount"] = np.nan
partner.loc[partner[~app].sample(frac=0.01, random_state=4).index, "amount"] = np.nan
partner.loc[partner.sample(5, random_state=5).index, "price"] = np.nan
partner.loc[partner.sample(1, random_state=6).index, "quantity"] = 200
partner.loc[partner.sample(1, random_state=7).index, "amount"] = 50_000_000
print("옷장패션 데이터 준비 완료:", partner.shape)


# 시나리오 1 — 진단
print("shape:", partner.shape)
partner.info()
print(partner.describe())

# 결측 진단
print("[열별 결측]")
print(missing_summary(partner))

amt_null = partner[partner["amount"].isnull()]
print("\n[amount 결측 행의 채널 분포]")
print(amt_null["channel"].value_counts(normalize=True).round(2))
print("\n[전체 채널 분포]")
print(partner["channel"].value_counts(normalize=True).round(2))

# IQR 이상치 — 수치형 컬럼 일괄 점검
num_cols = ["customer_age", "price", "quantity", "amount", "return_amount"]
print("[IQR 기준 이상치 개수]")
for c in num_cols:
    mask, (lo, up) = detect_outliers_iqr(partner[c].dropna())
    print(f"  {c:15s}  하한={lo:>12.1f}  상한={up:>12.1f}  이상치={mask.sum()}건")


# 시나리오 3 — 처리 코드
partner_clean = partner.copy()

# 1) customer_age 물리적 불가능 값 → NaN → 중앙값 대체
unrealistic = (partner_clean["customer_age"] < 1) | (partner_clean["customer_age"] > 110)
partner_clean.loc[unrealistic, "customer_age"] = np.nan
partner_clean["customer_age"] = partner_clean["customer_age"].fillna(
    partner_clean["customer_age"].median()
).astype(int)

# 2) quantity 이상치 → NaN → 중앙값 대체
mask_q, _ = detect_outliers_iqr(partner_clean["quantity"])
partner_clean.loc[mask_q, "quantity"] = np.nan
partner_clean["quantity"] = partner_clean["quantity"].fillna(
    partner_clean["quantity"].median()
).astype(int)

# 3) amount 이상치(50,000,000) → 유지 + 플래그
mask_a, _ = detect_outliers_iqr(partner_clean["amount"])
partner_clean["amount_outlier"] = mask_a.astype(int)

# 4) amount 결측 → 채널별 중앙값 대체 (MAR 가설)
partner_clean["amount"] = partner_clean["amount"].fillna(
    partner_clean.groupby("channel")["amount"].transform("median")
)

# 5) price 결측 → 카테고리별 중앙값 대체
partner_clean["price"] = partner_clean["price"].fillna(
    partner_clean.groupby("category")["price"].transform("median")
)

# 검증 출력
print("[처리 전 후 결측 비교]")
before = partner.isnull().sum()
after = partner_clean[partner.columns].isnull().sum()
print(pd.DataFrame({"before": before, "after": after}))

print("\n[처리 후 customer_age 범위]:",
      partner_clean["customer_age"].min(), "~", partner_clean["customer_age"].max())
print("[amount_outlier=1 건수]:", partner_clean["amount_outlier"].sum())


120
    order_id    amount
1     O00002  149700.0
74    O00075  149700.0
85    O00086   19800.0
97    O00098   19800.0
108   O00109   19800.0
건수: 476
(476, 7)
        missing  missing_pct(%)
amount       51            2.55
결측 행 채널 분포:
channel
app    0.8
web    0.2
Name: proportion, dtype: float64

전체 채널 분포:
channel
app    0.51
web    0.49
Name: proportion, dtype: float64
[원본 — 결측 제외 평균]
channel
app    88388.0
web    91980.0
Name: amount, dtype: float64
[A — 전체 중앙값 대체 후]
channel
app    87245.0
web    91648.0
Name: amount, dtype: float64
[B — 채널별 중앙값 대체 후]
channel
app    87245.0
web    91648.0
Name: amount, dtype: float64
하한: -120600 / 상한: 280200
이상치: 46건 (2.30%)
    order_id customer_id product_id category  quantity    amount channel
71    O00072       C0126       P038       가전         3  389700.0     app
115   O00116       C0019       P030       가전         3  389700.0     web
133   O00134       C0003       P033       뷰티         3  389700.0     web
154   O00155       C0051       P019   